In [ ]:
import numpy as np 
import pandas as pd 
import os 
import sys 
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt 
# %matplotlib notebook
sys.path.insert(1,'../../../../arimaclust/')
from arimaclust.models.centroid import ConsistentARMAClustering
from arimaclust.models.neighbours import NearestNeighbourModel
from arimaclust.autoarima.estimation import estimate_processes
from arimaclust.metrics.matrix import dist_matrix_parallel

In [ ]:
df_amd = pd.read_csv('data/amd/final.csv',index_col='Unnamed: 0')

In [ ]:
df_amd.drop(columns=['SLE','VES','BYN','AMD'],inplace=True) # no data

In [ ]:
from pmdarima.preprocessing import BoxCoxEndogTransformer


def prepare_data(df,bc=True,ws=None):
    X = []
    for col in df.columns:
        if ws:
            x = df[col].rolling(ws).mean().dropna().values
        else:
            x = df[col].dropna().values
        if bc:
            print(col)
            x, _ = BoxCoxEndogTransformer(lmbda2=1e-6).fit_transform(x)

        X.append(x)
    return X

def plot_mds(dist_matrix,labels,names):
    from sklearn import manifold 

    mds = manifold.MDS(
        n_components=2,
        max_iter=3000,
        eps=1e-9,
        random_state=42,
        dissimilarity="precomputed",
        n_jobs=-1,
        normalized_stress="auto",
    )
    pos = mds.fit(dist_matrix).embedding_


    nmds = manifold.MDS(
        n_components=2,
        metric=False,
        max_iter=3000,
        eps=1e-12,
        dissimilarity="precomputed",
        random_state=42,
        n_jobs=-1,
        n_init=1,
        normalized_stress="auto",
    )
    pos = nmds.fit_transform(dist_matrix, init=pos)
    fig, ax = plt.subplots(figsize=(12,8))
    ax.scatter(pos[:,0],pos[:,1],c=labels)
    for i in range(len(names)):
        ax.annotate(names[i], (pos[i][0], pos[i][1]),fontsize=8)
        
def plot_clusters(df,labels,k):
    for i in range(k):
        df[df.columns[labels==i]].plot()
        plt.legend(bbox_to_anchor=(0, 1, 1, 0), ncol=2)
        plt.show()

# 2016-2018 2 day average

In [ ]:
df_amd_1 = df_amd['2016-01-01':'2017-01-01']

In [ ]:
X = prepare_data(df=df_amd_1,bc=False,ws=2)

In [ ]:
process_args = {'max_p':3,
               'max_q':3,
               'trace':True,
               'd':0
              }

metric_args = {'tau':30}

In [ ]:
processes = estimate_processes(X=X,underlying_process='ARMA',**process_args)

In [ ]:
dist_matrix = dist_matrix_parallel(processes=processes,metric_name='piccolo',**metric_args)

In [ ]:
from sklearn import cluster

_, labels = cluster.affinity_propagation(dist_matrix,preference=5, random_state=0)
n_labels = labels.max() + 1

In [ ]:
n_labels

In [ ]:
plot_mds(dist_matrix=dist_matrix,labels=labels,names=df_amd_1.columns)

In [ ]:
model_amd = ConsistentARMAClustering(k=8, metric = 'precomputed', metric_args=metric_args,process_args=process_args)
model_amd.fit(dist_matrix)

In [ ]:
plot_mds(dist_matrix=dist_matrix,labels=model_amd.labels_,names=df_amd_1.columns)

In [ ]:
df_amd_scaled = pd.DataFrame(data=StandardScaler().fit_transform(df_amd_1),columns=df_amd_1.columns)
plot_clusters(df_amd_scaled,model_amd.labels_,k=10)

## 2017-2018

In [ ]:
df_amd_2 = df_amd['2017-01-01':'2018-01-01']
X_2 = prepare_data(df=df_amd_2,bc=False,ws=2)

process_args = {'max_p':3,
               'max_q':3,
               'trace':True,
               'd':0
              }

metric_args = {'tau':30}

processes_2 = estimate_processes(X=X_2,underlying_process='ARMA',**process_args)

dist_matrix_2 = dist_matrix_parallel(processes=processes_2,metric_name='piccolo',**metric_args)



In [ ]:
model_amd_2 = ConsistentARMAClustering(k=8, metric = 'precomputed', metric_args=metric_args,process_args=process_args)
model_amd_2.fit(dist_matrix_2)

In [ ]:
plot_mds(dist_matrix=dist_matrix_2,labels=model_amd_2.labels_,names=df_amd_2.columns)

In [ ]:
df_amd_scaled_2 = pd.DataFrame(data=StandardScaler().fit_transform(df_amd_2),columns=df_amd_2.columns)
plot_clusters(df_amd_scaled_2,model_amd_2.labels_,k=10)

In [ ]:
from arimaclust.tracking.mapping import mapp_hungarian

In [ ]:
r,c = mapp_hungarian(model_amd.labels_,model_amd_2.labels_)

In [ ]:
for i,j in zip(r,c):
    print(i,j)
    print(df_amd_1.columns[model_amd.labels_ == i],df_amd_2.columns[model_amd_2.labels_ == j], '\n')

## Log returns

In [ ]:
df_amd_1 = df_amd['2016-01-01':'2017-01-01']
df_amd_1 = np.log1p(df_amd_1.pct_change())
X_1 = prepare_data(df=df_amd_1,bc=False,ws=2)

process_args = {'max_p':3,
               'max_q':3,
               'trace':True,
               'd':0
              }

metric_args = {'tau':30}

processes_1 = estimate_processes(X=X_1,underlying_process='ARMA',**process_args)

dist_matrix_1 = dist_matrix_parallel(processes=processes_1,metric_name='piccolo',**metric_args)

In [ ]:
model_amd_1 = ConsistentARMAClustering(k=8, metric = 'precomputed', metric_args=metric_args,process_args=process_args)
model_amd_1.fit(dist_matrix_1)

In [ ]:
plot_mds(dist_matrix=dist_matrix_1,labels=model_amd_1.labels_,names=df_amd_1.columns)

In [ ]:
df_amd_2 = df_amd['2017-01-01':'2018-01-01']
df_amd_2 = np.log1p(df_amd_2.pct_change())

X_2 = prepare_data(df=df_amd_2,bc=False,ws=2)

process_args = {'max_p':3,
               'max_q':3,
               'trace':True,
               'd':0
              }

metric_args = {'tau':30}

processes_2 = estimate_processes(X=X_2,underlying_process='ARMA',**process_args)

dist_matrix_2 = dist_matrix_parallel(processes=processes_2,metric_name='piccolo',**metric_args)

In [ ]:
plot_mds(dist_matrix=dist_matrix_2,labels=model_amd_2.labels_,names=df_amd_2.columns)

In [ ]:
r,c = mapp_hungarian(model_amd_1.labels_,model_amd_2.labels_)
for i,j in zip(r,c):
    print(i,j)
    print(df_amd_1.columns[model_amd_1.labels_ == i],df_amd_2.columns[model_amd_2.labels_ == j], '\n')

## Continue

In [ ]:
model_amd = ConsistentARMAClustering(k=20,metric_args=metric_args,process_args=process_args)
model_amd.fit(X)

In [ ]:
mask = np.ones(model_amd.dist_matrix.shape, dtype=bool)
np.fill_diagonal(mask, 0)
print(f'Min distance (ignore diag): {model_amd.dist_matrix[mask].min()}, max distance: {model_amd.dist_matrix[mask].max()} ')


In [ ]:
plot_mds(dist_matrix=model_amd.dist_matrix,labels=model_amd.labels_,names=df_amd_1.columns)

In [ ]:
df_amd_scaled = pd.DataFrame(data=StandardScaler().fit_transform(df_amd_1),columns=df_amd_1.columns)
plot_clusters(df_amd_scaled,model_amd.labels_,k=20)

In [ ]:
df_amd_scaled[['EUR','CHF','USD','RUB']].plot()

In [ ]:
def get_symb_dists(dist_matrix,df,s1,s2):
    return dist_matrix[list(df.columns).index(s1)][list(df.columns).index(s2)]

In [ ]:
s1 = 'EUR'
s2 = 'CHF'
print(f'd({s1},{s2}) = {get_symb_dists(model_amd.dist_matrix,df_amd_1,s1,s2)}')

In [ ]:
s1 = 'EUR'
s2 = 'USD'
print(f'd({s1},{s2}) = {get_symb_dists(model_amd.dist_matrix,df_amd_1,s1,s2)}')

In [ ]:
s1 = 'EUR'
s2 = 'GBP'
print(f'd({s1},{s2}) = {get_symb_dists(model_amd.dist_matrix,df_amd_1,s1,s2)}')

In [ ]:
df_amd_scaled[['EUR','RUB','GBP']].plot()

In [ ]:
# alpha = 0.05
nn_model = NearestNeighbourModel(alpha=0.1,metric='precomputed')
nn_model.fit(model_amd.dist_matrix)

In [ ]:
plot_clusters(df_amd_scaled,nn_model.labels_,k=nn_model.labels_.max().astype(int))

# 2016-2018 2 differencing

In [ ]:
X = prepare_data(df=df_amd_1,bc=False,ws=None)

In [ ]:
process_args = {'max_p':3,
               'max_q':3,
                'start_p':0,
                'start_q':0,
                'd':0,
                'seasonal':False,
                'out_of_sample_size':100
              }

metric_args = {'tau':15}

In [ ]:
model_amd = ConsistentARMAClustering(k=20,metric_args=metric_args,process_args=process_args)
model_amd.fit(X)

In [ ]:
mask = np.ones(model_amd.dist_matrix.shape, dtype=bool)
np.fill_diagonal(mask, 0)
print(f'Min distance (ignore diag): {model_amd.dist_matrix[mask].min()}, max distance: {model_amd.dist_matrix[mask].max()} ')


In [ ]:
plot_mds(dist_matrix=model_amd.dist_matrix,labels=model_amd.labels_,names=df_amd_1.columns)

In [ ]:
plot_clusters(df_amd_scaled,model_amd.labels_,k=20)